# NRPy in Ten Minutes

Build one small NRPy2 workflow: define a wave-energy diagnostic, generate the
matching C assignment, register a callable C function, and validate that the
stored artifacts preserve the intended formula.

Navigation: [Index](../index.ipynb) |
Previous: [Getting Started with NRPy](../0-getting_started/install_and_run.ipynb) |
Next: [Parameters](params.ipynb)

## Learning Goals

- Explain why NRPy2 turns symbolic formulas into generated C code.
- Connect one scalar-wave diagnostic to NRPy2 symbols, parameters, and fields.
- Validate that one symbolic formula drives both the generated assignment and
  registered C function.

## Problem Statement

For a scalar wave, let `uu` represent the field value $u$, `vv` represent its
time derivative $v$, and `overview_wave_speed` represent the wave speed $c$.
This notebook follows one diagnostic through NRPy2:

$$
E = \frac{1}{2}\left(v^2 + c^2 u^2\right).
$$

The diagnostic is small, but it shows the larger pattern: keep the mathematics
in symbolic form, emit generated C code from that symbolic form, and check the
stored generated artifacts.

## Words for This Notebook

- **Symbolic expression:** the Python representation of a formula that NRPy2 can
  later translate into C.
- **Grid field:** a named quantity stored at each grid point; here `uu` stores
  the wave value and `vv` stores its time derivative.
- **Code parameter:** a named value that appears in generated C code; here
  `overview_wave_speed` stands for the wave speed in generated project code.
- **Generated C assignment:** one C statement emitted from a symbolic
  expression.
- **Stored C function:** a complete C function kept in NRPy2's in-memory
  function list so project writers can include it later.


## Why NRPy2 Generates Code

Hand-translating equations into C is easy for a single line and brittle for a
large numerical relativity system. NRPy2 lets the notebook keep the mathematics
as symbolic Python objects, then emits C code from those objects.

The rest of this notebook checks that the same wave-energy formula controls the
printed assignment and the registered C function.

## Setup: Import Tools and Clear Tutorial State

The setup cell imports SymPy, C code generation, C function registration, grid
field registration, and code parameters. It also removes only tutorial-owned
state so rerunning the notebook starts from the same names.

In [1]:
import sympy as sp

import nrpy.c_codegen as ccg
from nrpy.c_function import CFunction_dict, register_CFunction
import nrpy.grid as grid
import nrpy.params as par

removed_functions = []
for name in ["overview_energy_density"]:
    if CFunction_dict.pop(name, None) is not None:
        removed_functions.append(name)

removed_grid_fields = []
for name in ["uu", "vv"]:
    if grid.glb_gridfcs_dict.pop(name, None) is not None:
        removed_grid_fields.append(name)

par.glb_code_params_dict.pop("overview_wave_speed", None)
par.set_parval_from_str("Infrastructure", "BHaH")

print("cleared functions:", removed_functions)
print("cleared grid fields:", removed_grid_fields)
print("infrastructure:", par.parval_from_str("Infrastructure"))

cleared functions: []
cleared grid fields: []
infrastructure: BHaH


## Step 1: Build the Symbolic Diagnostic and Assignment

The table maps the mathematical names to the NRPy2 names used below.

| Mathematical object | NRPy2 name | Role in generated code |
| --- | --- | --- |
| $u$ | `uu` | grid field read |
| $v$ | `vv` | grid field read |
| $c$ | `overview_wave_speed` | code parameter |
| $E$ | `energy_density` | assignment target |

The code cell registers the field and parameter names, builds the symbolic
formula, and emits the generated C assignment.

In [2]:
uu, vv = grid.register_gridfunctions(["uu", "vv"], group="EVOL")
wave_speed = par.register_CodeParameter(
    "REAL", "tutorial_overview", "overview_wave_speed", 1.0
)
energy_density = sp.Rational(1, 2) * (vv**2 + wave_speed**2 * uu**2)
assignment = ccg.c_codegen(
    energy_density,
    "energy_density",
    include_braces=False,
    verbose=False,
)

print("symbolic expression:")
print(energy_density)
print("generated assignment:")
print(assignment)

symbolic expression:
overview_wave_speed**2*uu**2/2 + vv**2/2
generated assignment:
energy_density = (1.0/2.0)*((overview_wave_speed)*(overview_wave_speed))*((uu)*(uu)) + (1.0/2.0)*((vv)*(vv));



## Step 2: Generate the Function Body from the Same Formula

A standalone C function receives the wave speed as an argument named
`wave_speed`, while generated project code uses the code parameter name
`overview_wave_speed`. The next cell substitutes only that name and generates
the function-body assignment from the same symbolic formula.

In [3]:
function_wave_speed = sp.Symbol("wave_speed", real=True)
function_energy_density = energy_density.subs(wave_speed, function_wave_speed)
function_body_assignment = ccg.c_codegen(
    function_energy_density,
    "*energy",
    include_braces=False,
    verbose=False,
)
function_body = function_body_assignment.strip() + "\nreturn;"

print("function symbolic expression:")
print(function_energy_density)
print("generated function body:")
print(function_body)


function symbolic expression:
uu**2*wave_speed**2/2 + vv**2/2
generated function body:
*energy = (1.0/2.0)*((uu)*(uu))*((wave_speed)*(wave_speed)) + (1.0/2.0)*((vv)*(vv));
return;


## Step 3: Register the Callable C Function

The stored function list keeps the generated body together with the function
name, return type, and input list. A later project writer can emit the stored
function to source files.


In [4]:
register_CFunction(
    name="overview_energy_density",
    desc="Compute a compact wave-energy diagnostic.",
    cfunc_type="void",
    params=(
        "const double uu, const double vv, "
        "const double wave_speed, double *energy"
    ),
    body=function_body,
)
stored_function = CFunction_dict["overview_energy_density"]
print("registered keys:", sorted(CFunction_dict))
print("stored function declaration:")
print(stored_function.function_prototype)


registered keys: ['overview_energy_density']
stored function declaration:
void overview_energy_density(const double uu, const double vv, const double wave_speed, double *energy);


## Validation Check

A valid miniature workflow must preserve the symbolic formula in both generated
C artifacts and register the function with the expected metadata.

In [5]:
expected_energy_density = sp.Rational(1, 2) * (
    vv**2 + wave_speed**2 * uu**2
)
if sp.simplify(energy_density - expected_energy_density) != 0:
    raise RuntimeError(f"Unexpected symbolic expression: {energy_density}")

expected_function_energy_density = expected_energy_density.subs(
    wave_speed, function_wave_speed
)
if sp.simplify(function_energy_density - expected_function_energy_density) != 0:
    raise RuntimeError(f"Unexpected function expression: {function_energy_density}")

for expected_piece in ["energy_density =", "overview_wave_speed", "uu", "vv"]:
    if expected_piece not in assignment:
        raise RuntimeError(f"Missing {expected_piece} in generated assignment")
if "overview_wave_speed" in function_body:
    raise RuntimeError("Function body should use its wave_speed argument.")
if "wave_speed" not in function_body:
    raise RuntimeError("Function body is missing the wave_speed argument.")

stored_function = CFunction_dict.get("overview_energy_density")
if stored_function is None:
    raise RuntimeError("overview_energy_density was not stored.")
if stored_function.name != "overview_energy_density":
    raise RuntimeError(f"Unexpected function name: {stored_function.name}")
if stored_function.cfunc_type != "void":
    raise RuntimeError(f"Unexpected function type: {stored_function.cfunc_type}")
expected_params = (
    "const double uu, const double vv, "
    "const double wave_speed, double *energy"
)
if stored_function.params != expected_params:
    raise RuntimeError(f"Unexpected function parameters: {stored_function.params}")
if stored_function.body != function_body:
    raise RuntimeError("Stored function body does not match generated body.")

print("validated symbolic expression:", energy_density)
print("validated assignment target:", "energy_density")
print("validated generated function body:", function_body.splitlines()[0])
print("validated stored function:", stored_function.name)

validated symbolic expression: overview_wave_speed**2*uu**2/2 + vv**2/2
validated assignment target: energy_density
validated generated function body: *energy = (1.0/2.0)*((uu)*(uu))*((wave_speed)*(wave_speed)) + (1.0/2.0)*((vv)*(vv));
validated stored function: overview_energy_density


## Learning Check

After running the validation cell, identify which names are stored by NRPy2 and
which names exist only as local Python variables in this notebook. Then explain
why the generated assignment uses `overview_wave_speed` while the registered C
function body uses `wave_speed`.

## Continue to Parameters

- [Parameters](params.ipynb)
- [Indexed Expressions](indexedexp.ipynb)
- [Grid Fields and Grid Metadata](grid.ipynb)
- [C Code Generation](c_codegen.ipynb)
- [C Function Registration](c_function.ipynb)
- [Wave Equation and C Code Generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb)